# Knee Osteoarthritis Classification using SE-ResNeXt-50 (Paper Method)

This notebook implements the training method and model architecture from the paper **"Multimodal Machine Learning-based Knee osteoarthritis progression prediction from plain Radiographs and clinical Data"** using an **SE-ResNeXt-50-32x4d** backbone.

### Techniques Applied from the Paper:
1. **Model Architecture**:
   - Backbone: `se-resnext50_32x4d` initialized with ImageNet pre-trained weights.
   - Classifier: Dropout ($p = 0.5$) followed by a Fully Connected (FC) layer with 5 outputs to predict the baseline KL grade.
2. **Two-Stage Training Strategy**:
   - **Stage 1 (FC Warm-up / Backbone Frozen)**: The weights of the Conv layers (backbone) are frozen for the first 2 epochs. Only the classifier heads are trained.
   - **Stage 2 (Backbone Unfrozen)**: All layers of the network are unfrozen and trained for 20 epochs.
3. **Optimizer & LR Scheduler**:
   - Optimizer: Adam.
   - Weight decay: $1e-4$.
   - Learning rate: Initial learning rate of $1e-3$, which is dropped by a factor of 10 (to $1e-4$) at the 15th epoch (overall epoch index 14).
4. **Data Augmentation**:
   - Random horizontal flip.
   - Random rotation $\pm 5$ degrees.
   - Random cropping from $310 \times 310$ to $300 \times 300$.
   - Random noise addition (Gaussian noise).
   - Random gamma correction.
5. **Inference & Evaluation**:
   - **5-crop Test-Time Augmentation (TTA)**: 4 corners + 1 center crop of size $300 \times 300$ from the $310 \times 310$ image, with predictions averaged over the 5 crops.

## 0. Import lib
Import library, load device, connect to google drive

In [ ]:
import os
import random
import hashlib
from collections import Counter
from typing import List, Union
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import tqdm
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

# Try mounting drive (if on Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully.")
except ImportError:
    print("Not running in Google Colab. Skipping Drive mount.")

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

# Install timm if needed
try:
    import timm
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "-q", "timm"])
    import timm

# Install torchmetrics if needed
try:
    import torchmetrics
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "-q", "torchmetrics"])
    import torchmetrics

# Install seaborn if needed
try:
    import seaborn
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "-q", "seaborn"])
    import seaborn

## 1. Prepare dataset 
unzip dataset from google drive

In [ ]:
import subprocess
import os
import torch
import numpy as np

# Unzip dataset from Drive if running on Google Colab
dataset_zip = "/content/drive/MyDrive/Datasets/kaggle_knee_osteoarthritis.zip"
if os.path.exists(dataset_zip):
    print("Unzipping dataset from Google Drive...")
    subprocess.run(["unzip", "-q", dataset_zip, "-d", "/content/Datasets"])
else:
    print("Zip file not found at default Drive path. Assuming local dataset path.")

# =========================================================================
# CONFIGURATION & PARAMETERS (Unified Input Config)
# =========================================================================
class TrainingConfig:
    # Model Selection
    model_name = "seresnext50_32x4d"
    pretrained = True
    dropout_rate = 0.5
    
    # Dataset & Paths
    dataset_root = "/content/Datasets/kaggle_knee_osteoarthritis"
    checkpoint_dir = "/content/drive/MyDrive/Models/se_resnext50_checkpoints"
    img_size = 310          # Paper specifies 310x310 input spacing
    crop_size = 300         # Paper specifies random cropping to 300x300
    batch_size = 16         # Paper uses 64, but we default to 16 to avoid OOM on normal GPUs
    seed = 42
    
    # Imbalance Handling
    use_balanced_sampler = True
    
    # Inference Options
    use_tta = True          # Enables 5-crop Test-Time Augmentation as described in the paper
    
    # Training Stage & Freezing Options
    # Options: "paper_2stage", "3-stage", "2-stage", "standard"
    # "paper_2stage" implements the paper's strategy: Stage 1 (Backbone frozen, 2 epochs), Stage 2 (Unfrozen, 20 epochs)
    training_pipeline = "paper_2stage"
    resume_from_last = True
    
    # Phase Epochs
    stage1_epochs = 2       # Paper: 2 epochs
    stage2_epochs = 20      # Paper: 20 epochs
    stage3_epochs = 0       # Not used in paper_2stage
    total_epochs_standard = 22 # Only used if training_pipeline is "standard"
    
    # Learning Rates
    lr_warmup = 1e-3        # Paper: 1e-3
    lr_coarse_head = 1e-3
    lr_coarse_backbone = 1e-3
    lr_finetune = 1e-3      # Paper: 1e-3 initial learning rate
    lr_standard = 1e-3      # If pipeline is "standard"
    
    weight_decay = 1e-4     # Paper: 1e-4
    
    # Loss Function Choices
    # Options: "ce" (Cross-Entropy), "corn" (Conditional Ordinal), "coral", "focal_corn"
    loss_stage1 = "ce"
    loss_stage2 = "ce"
    loss_stage3 = "ce"
    loss_standard = "ce"
    
    # Learning Rate Schedulers
    # Options: "paper_step" (LR dropped at epoch 15), "cosine", "step", "none"
    scheduler_stage2 = "paper_step"
    scheduler_stage3 = "none"
    scheduler_standard = "paper_step"

def log_config(config):
    print("="*65)
    print(" ACTIVE TRAINING CONFIGURATION LOG")
    print("="*65)
    attrs = [attr for attr in dir(config) if not attr.startswith('__') and not callable(getattr(config, attr))]
    for attr in attrs:
        print(f"{attr:<25} : {getattr(config, attr)}")
    print("="*65)

# Log configurations
log_config(TrainingConfig)

# Set global variables
DATASET_ROOT_PATH = TrainingConfig.dataset_root
CHECKPOINT_SAVE_DIR = TrainingConfig.checkpoint_dir
BATCH_SIZE = TrainingConfig.batch_size
IMG_SIZE = TrainingConfig.img_size
CROP_SIZE = TrainingConfig.crop_size

# Set random seed
torch.manual_seed(TrainingConfig.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(TrainingConfig.seed)
np.random.seed(TrainingConfig.seed)
import random
random.seed(TrainingConfig.seed)

os.makedirs(CHECKPOINT_SAVE_DIR, exist_ok=True)

## 2. Train config loader
Load train config from input user + seed

In [ ]:
# Re-log configuration to avoid redefinition and early stopping bugs
log_config(TrainingConfig)

DATASET_ROOT_PATH = TrainingConfig.dataset_root
CHECKPOINT_SAVE_DIR = TrainingConfig.checkpoint_dir
BATCH_SIZE = TrainingConfig.batch_size
IMG_SIZE = TrainingConfig.img_size
CROP_SIZE = TrainingConfig.crop_size

torch.manual_seed(TrainingConfig.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(TrainingConfig.seed)
np.random.seed(TrainingConfig.seed)
random.seed(TrainingConfig.seed)

os.makedirs(CHECKPOINT_SAVE_DIR, exist_ok=True)

## 3. Preprocessing image
Padding + CLAHE + Transforms (train, val, minority) + Remove duplicate

In [ ]:
class SquarePadOpenCV(object):
    """Pads a rectangular image to a square."""
    def __call__(self, image):
        h, w = image.shape[:2]
        max_wh = max(h, w)
        pad_top = (max_wh - h) // 2
        pad_bottom = max_wh - h - pad_top
        pad_left = (max_wh - w) // 2
        pad_right = max_wh - w - pad_left
        
        padded_image = cv2.copyMakeBorder(
            image, pad_top, pad_bottom, pad_left, pad_right, 
            borderType=cv2.BORDER_CONSTANT, value=[0, 0, 0]
        )
        return padded_image

class OpenCVCLAHE(object):
    """Applies CLAHE (Contrast Limited Adaptive Histogram Equalization) using OpenCV."""
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size

    def __call__(self, img_rgb: np.ndarray) -> np.ndarray:
        clahe = cv2.createCLAHE(clipLimit=self.clip_limit, tileGridSize=self.tile_grid_size)
        img_lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
        l_channel, a_channel, b_channel = cv2.split(img_lab)
        clahe_l_channel = clahe.apply(l_channel)
        merged_lab_image = cv2.merge((clahe_l_channel, a_channel, b_channel))
        return cv2.cvtColor(merged_lab_image, cv2.COLOR_LAB2RGB)

class AddGaussianNoise(object):
    """Adds random Gaussian noise to a PyTorch tensor."""
    def __init__(self, mean=0.0, std=0.01, p=0.5):
        self.mean = mean
        self.std = std
        self.p = p

    def __call__(self, tensor: torch.Tensor) -> torch.Tensor:
        if random.random() < self.p:
            noise = torch.randn(tensor.size()) * self.std + self.mean
            return tensor + noise
        return tensor

class RandomGammaCorrection(object):
    """Applies random gamma correction to a PIL image or PyTorch tensor."""
    def __init__(self, gamma_range=(0.7, 1.3), p=0.5):
        self.gamma_range = gamma_range
        self.p = p

    def __call__(self, img):
        if random.random() < self.p:
            gamma = random.uniform(*self.gamma_range)
            import torchvision.transforms.functional as TF
            return TF.adjust_gamma(img, gamma)
        return img

def get_transforms(img_size=310, crop_size=300):
    """Returns training, validation, and TTA validation transforms."""
    # Training transform with Paper-specified Augmentations
    train_transform = transforms.Compose([
        SquarePadOpenCV(),
        OpenCVCLAHE(),
        transforms.ToPILImage(),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=5),             # Paper: ±5 degrees
        transforms.Resize((img_size, img_size)),         # Resize to 310x310
        RandomGammaCorrection(gamma_range=(0.7, 1.3), p=0.5), # Paper: random gamma correction
        transforms.RandomCrop(crop_size),                # Paper: random cropping to 300x300
        transforms.ToTensor(),
        AddGaussianNoise(mean=0.0, std=0.01, p=0.5),      # Paper: random noise addition
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # Standard Validation Transform (Center Crop to 300x300)
    val_transform = transforms.Compose([
        SquarePadOpenCV(),
        OpenCVCLAHE(),
        transforms.ToPILImage(),
        transforms.Resize((img_size, img_size)),
        transforms.CenterCrop(crop_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # 5-Crop Test-Time Augmentation (TTA) Transform
    val_transform_tta = transforms.Compose([
        SquarePadOpenCV(),
        OpenCVCLAHE(),
        transforms.ToPILImage(),
        transforms.Resize((img_size, img_size)),
        transforms.FiveCrop(crop_size),                  # Paper: 5 crops of 300x300
        transforms.Lambda(lambda crops: torch.stack([
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])(transforms.ToTensor()(crop))
            for crop in crops
        ]))
    ])

    return train_transform, val_transform, val_transform_tta

def remove_duplicate_images(image_paths: list, labels: list, exclude_hashes: set = None):
    """Removes duplicate images using MD5 hashing."""
    total_found = len(image_paths)
    unique_paths, unique_labels, unique_hashes = [], [], set()
    internal_dup_count, leakage_count = 0, 0
    
    for path, label in zip(image_paths, labels):
        hash_md5 = hashlib.md5()
        try:
            with open(path, "rb") as f:
                for chunk in iter(lambda: f.read(4096), b""): 
                    hash_md5.update(chunk)
            h = hash_md5.hexdigest()
        except Exception as e:
            print(f"Warning: Could not read image {path}: {e}")
            continue
            
        if exclude_hashes and h in exclude_hashes:
            leakage_count += 1
            continue
        if h in unique_hashes:
            internal_dup_count += 1
            continue
            
        unique_hashes.add(h)
        unique_paths.append(path)
        unique_labels.append(label)
        
    print(f"\n--- Deduplication: Files found: {total_found} | Unique kept: {len(unique_paths)} | Dupes removed: {internal_dup_count} | Cross-split leaks: {leakage_count}")
    return unique_paths, unique_labels, unique_hashes

## 4. Dataset
Load kaggle dataset (apply transform + duplicate remove)

In [ ]:
class KaggleKneeOsteoarthritisDataset(Dataset):
    """Dataset class for loading Kaggle Knee OA dataset splits."""
    def __init__(self, root: str, split_dir: str, transform=None, exclude_hashes: set = None):
        self.root = root
        self.transform = transform
        self.exclude_hashes = exclude_hashes
        raw_paths, raw_labels = [], []
        split_path = os.path.join(root, split_dir)
        
        if not os.path.isdir(split_path): 
            raise FileNotFoundError(f"Split directory not found: {split_path}")
            
        class_names = sorted([d for d in os.listdir(split_path) if os.path.isdir(os.path.join(split_path, d)) and d.isdigit()])
        print(f"Loading '{split_dir}' split from: {split_path}")
        
        for class_name in class_names:
            class_dir = os.path.join(split_path, class_name)
            label = int(class_name)
            valid_extensions = ('.png', '.jpg', '.jpeg')
            image_files = [f for f in os.listdir(class_dir) if f.lower().endswith(valid_extensions)]
            for file_name in image_files:
                raw_paths.append(os.path.join(class_dir, file_name))
                raw_labels.append(label)
                
        self.image_paths, self.labels, self.image_hashes = remove_duplicate_images(
            raw_paths, raw_labels, exclude_hashes=self.exclude_hashes
        )

    def load_image_from_path(self, image_path: str) -> np.ndarray:
        img_bgr = cv2.imread(image_path)
        if img_bgr is None: 
            raise IOError(f"Could not read image: {image_path}")
        return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    def __getitem__(self, idx: int):
        image = self.load_image_from_path(self.image_paths[idx])
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

    def __len__(self) -> int: 
        return len(self.image_paths)

## 5. Dataloader
Prepare train, val dataloader

In [ ]:
# Create transforms
train_transform, val_transform, val_transform_tta = get_transforms(img_size=IMG_SIZE, crop_size=CROP_SIZE)

# Choose validation transform based on TTA flag
val_loader_transform = val_transform_tta if TrainingConfig.use_tta else val_transform

# Load training dataset
train_dataset = KaggleKneeOsteoarthritisDataset(
    root=DATASET_ROOT_PATH, split_dir="train", transform=train_transform
)
train_hashes = set(train_dataset.image_hashes)

# Load validation dataset
val_split_dir = "val" if os.path.isdir(os.path.join(DATASET_ROOT_PATH, "val")) else "test"
val_dataset = KaggleKneeOsteoarthritisDataset(
    root=DATASET_ROOT_PATH, split_dir=val_split_dir, transform=val_loader_transform, exclude_hashes=train_hashes
)

# Imbalance handling: Calculate Class-Aware WeightedRandomSampler for training split
from torch.utils.data import WeightedRandomSampler

# Count samples of each class
class_counts = Counter(train_dataset.labels)
print(f"Training class distribution: {dict(sorted(class_counts.items()))}")

# Create training loader based on config
if TrainingConfig.use_balanced_sampler:
    print("Using WeightedRandomSampler for class balance.")
    class_weights = {cls: 1.0 / count for cls, count in class_counts.items()}
    sample_weights = [class_weights[label] for label in train_dataset.labels]
    sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)
    train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
else:
    print("Using standard shuffled DataLoader.")
    train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

val_loader = DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Data loaders ready. Train batches: {len(train_loader)} | Validation batches: {len(val_loader)}")

## 6. CORAL & CORN convert label
Normal Ce: 5 class -> CORAL: 4 class

In [ ]:
def label_to_levels(label, num_classes, dtype=torch.float32):
    batch_size = label.size(0)
    levels = torch.zeros(batch_size, num_classes - 1, dtype=dtype, device=label.device)
    for i in range(batch_size):
        levels[i, :label[i]] = 1.0
    return levels

def coral_loss(logits, y_train, num_classes=5):
    levels = label_to_levels(y_train, num_classes)
    loss = F.binary_cross_entropy_with_logits(logits, levels)
    return loss

def coral_label_from_logits(logits):
    probs = torch.sigmoid(logits)
    predicted = (probs > 0.5).sum(dim=1)
    return predicted
     
def corn_loss(logits, y_train, num_classes=5, task_weights=[2.0, 1.8, 1.2, 1.0]):
    loss = 0.0
    num_tasks = num_classes - 1
    for k in range(num_tasks):
        mask = y_train >= k
        if not mask.any():
            continue
        logits_k = logits[mask, k]
        targets_k = (y_train[mask] > k).float()
        # Apply label smoothing (0.1)
        targets_k = targets_k * (1 - 0.1) + (1 - targets_k) * 0.1
        w_k = task_weights[k] if k < len(task_weights) else 1.0
        loss += w_k * F.binary_cross_entropy_with_logits(logits_k, targets_k)
    return loss / num_tasks

def focal_corn_loss(logits, y_train, num_classes=5, gamma=2.0, alpha=0.25, task_weights=[2.0, 1.8, 1.2, 1.0]):
    loss = 0.0
    num_tasks = num_classes - 1
    for k in range(num_tasks):
        mask = y_train >= k
        if not mask.any():
            continue
        logits_k = logits[mask, k]
        targets_k = (y_train[mask] > k).float()
        # Apply label smoothing (0.1)
        targets_k = targets_k * (1 - 0.1) + (1 - targets_k) * 0.1
        
        bce = F.binary_cross_entropy_with_logits(logits_k, targets_k, reduction='none')
        p = torch.sigmoid(logits_k)
        p_t = p * targets_k + (1 - p) * (1 - targets_k)
        focal_weight = alpha * (1 - p_t) ** gamma
        
        w_k = task_weights[k] if k < len(task_weights) else 1.0
        loss += w_k * (focal_weight * bce).mean()
    return loss / num_tasks

def corn_probas(logits):
    cond_probas = torch.sigmoid(logits)
    batch_size = logits.size(0)
    num_classes = logits.size(1) + 1
    probas = torch.zeros(batch_size, num_classes, device=logits.device)
    cumprod = torch.cumprod(cond_probas, dim=1)
    probas[:, 0] = 1.0 - cond_probas[:, 0]
    for i in range(1, num_classes - 1):
        probas[:, i] = cumprod[:, i - 1] * (1.0 - cond_probas[:, i])
    probas[:, -1] = cumprod[:, -1]
    return probas

def corn_label_from_logits(logits):
    probas = corn_probas(logits)
    return torch.argmax(probas, dim=1)

## 7. Model
SE-ResNeXt-50 model with single head architecture for KL grading

In [ ]:
class SEResNeXt50Model(nn.Module):
    def __init__(self, num_classes: int = 5, pretrained: bool = True, loss_type: str = "ce"):
        super(SEResNeXt50Model, self).__init__()
        self.loss_type = loss_type
        self.num_classes = num_classes
        
        # Set head dimensions. CE has num_classes (5), Ordinal (CORN/CORAL) has num_classes - 1 (4)
        out_features_kl = num_classes if loss_type == "ce" else num_classes - 1
        
        # Backbone using se_resnext50_32x4d
        self.backbone = timm.create_model(
            'seresnext50_32x4d', 
            pretrained=pretrained,
            num_classes=0 # Returns features (pooled, dim 2048)
        )
        
        num_features = self.backbone.num_features
        
        # Single head architecture: predict baseline KL grade (Dropout rate 0.5)
        self.fc_kl = nn.Sequential(
            nn.Dropout(p=TrainingConfig.dropout_rate),
            nn.Linear(num_features, out_features_kl)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x)
        logits_kl = self.fc_kl(features)
        return logits_kl

    def freeze_backbone(self):
        print('Freezing SE-ResNeXt-50 backbone features.')
        for param in self.backbone.parameters():
            param.requires_grad = False

    def unfreeze_backbone(self):
        print('Unfreezing all SE-ResNeXt-50 parameters.')
        for param in self.parameters():
            param.requires_grad = True

    def fit(self, epoch, data_loader, optimizer, loss_type, device):
        self.to(device)
        self.train()
        running_loss, total, correct = 0.0, 0, 0
        
        criterion = get_loss_criterion(loss_type, num_classes=self.num_classes)
        predict_fn = get_prediction_helper(loss_type)
        
        progress_bar = tqdm.tqdm(data_loader, desc=f"Epoch {epoch+1} [TRAIN]")
        for images, labels in progress_bar:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = self(images)
            loss = criterion(outputs, labels)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.parameters(), max_norm=1.0)
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            
            predicted = predict_fn(outputs)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            progress_bar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "acc": f"{100.0 * correct / total:.2f}%"
            })
            
        return running_loss / total, 100.0 * correct / total

    def evaluate(self, epoch, data_loader, loss_type, device, description="VALIDATE"):
        self.to(device)
        self.eval()
        running_loss, total, correct = 0.0, 0, 0
        all_preds, all_labels, all_probas = [], [], []
        
        criterion = get_loss_criterion(loss_type, num_classes=self.num_classes)
        predict_fn = get_prediction_helper(loss_type)
        
        if loss_type == "ce":
            probas_fn = lambda x: F.softmax(x, dim=1)
        elif loss_type in ["corn", "focal_corn"]:
            probas_fn = corn_probas
        elif loss_type == "coral":
            probas_fn = torch.sigmoid
            
        progress_bar = tqdm.tqdm(data_loader, desc=f"Epoch {epoch+1} [{description}]" if epoch is not None else description)
        with torch.no_grad():
            for images, labels in progress_bar:
                labels = labels.to(device)
                
                # Check if DataLoader is returning 5 crops for TTA
                if len(images.shape) == 5: # Shape: [batch_size, 5, 3, crop_h, crop_w]
                    bs, n_crops, c, h, w = images.size()
                    flat_images = images.view(-1, c, h, w).to(device)
                    outputs = self(flat_images) # shape: [batch_size * 5, out_features]
                    
                    # Convert to probabilities first
                    probas = probas_fn(outputs) # shape: [batch_size * 5, classes]
                    # Average probabilities across the 5 crops
                    probas = probas.view(bs, n_crops, -1).mean(dim=1)
                    
                    # Average logits across the 5 crops for loss calculation
                    logits = outputs.view(bs, n_crops, -1).mean(dim=1)
                    loss = criterion(logits, labels)
                    
                    # Predict from averaged probabilities
                    if loss_type == "ce":
                        predicted = torch.argmax(probas, dim=1)
                    elif loss_type in ["corn", "focal_corn"]:
                        predicted = torch.argmax(probas, dim=1)
                    elif loss_type == "coral":
                        predicted = (probas > 0.5).sum(dim=1)
                else:
                    images = images.to(device)
                    outputs = self(images)
                    loss = criterion(outputs, labels)
                    probas = probas_fn(outputs)
                    predicted = predict_fn(outputs)
                
                running_loss += loss.item() * labels.size(0)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                all_probas.extend(probas.cpu().numpy())
                
        # Convert to numpy arrays for sklearn metrics
        all_labels_arr = np.array(all_labels)
        all_probas_arr = np.array(all_probas)
        
        # Pad probas array to 5 classes if it has 4 (for CORAL/CORN) during OVR calculations
        if all_probas_arr.shape[1] == 4:
            if loss_type in ["corn", "focal_corn"]:
                pass
            elif loss_type == "coral":
                temp = np.zeros((all_probas_arr.shape[0], 5))
                temp[:, 0] = 1.0 - all_probas_arr[:, 0]
                for idx in range(1, 4):
                    temp[:, idx] = all_probas_arr[:, idx - 1] - all_probas_arr[:, idx]
                temp[:, 4] = all_probas_arr[:, 3]
                all_probas_arr = np.clip(temp, 0.0, 1.0)
                
        report = classification_report(
            all_labels_arr, all_preds, 
            target_names=[str(i) for i in range(5)], 
            zero_division=0
        )
        
        # Calculate Cohen's Quadratic Weighted Kappa
        from torchmetrics.classification import CohenKappa
        kappa_metric = CohenKappa(task="multiclass", num_classes=5, weights="quadratic")
        kappa_score = kappa_metric(torch.tensor(all_preds), torch.tensor(all_labels)).item()
        
        # Calculate AUC and AP (One-vs-Rest Macro)
        from sklearn.metrics import roc_auc_score, average_precision_score
        try:
            auc_score = roc_auc_score(all_labels_arr, all_probas_arr, multi_class='ovr', average='macro')
        except Exception as e:
            auc_score = 0.0
            print(f"ROC AUC computation failed: {e}")
            
        try:
            y_one_hot = np.eye(5)[all_labels_arr]
            ap_score = average_precision_score(y_one_hot, all_probas_arr, average='macro')
        except Exception as e:
            ap_score = 0.0
            print(f"Average Precision (AP) computation failed: {e}")
            
        print(f"\nQuadratic Weighted Kappa (QWK): {kappa_score:.4f}")
        print(f"ROC AUC (OVR Macro): {auc_score:.4f}")
        print(f"Average Precision (AP Macro): {ap_score:.4f}")
        
        metrics = {
            "loss": running_loss / total,
            "acc": 100.0 * correct / total,
            "qwk": kappa_score,
            "auc": auc_score,
            "ap": ap_score,
            "probas": all_probas_arr,
            "report": report
        }
        return metrics

## 8. Prepare for pipeline
Prepare loss, prediction, lr scheduler, pipeline

In [ ]:
best_val_qwk = -1.0
best_val_qwk_stage2 = -1.0
history = []

def get_loss_criterion(loss_type, num_classes=5):
    if loss_type == "ce":
        return nn.CrossEntropyLoss()
    elif loss_type == "corn":
        return lambda logits, targets: corn_loss(logits, targets, num_classes)
    elif loss_type == "coral":
        return lambda logits, targets: coral_loss(logits, targets, num_classes)
    elif loss_type == "focal_corn":
        return lambda logits, targets: focal_corn_loss(logits, targets, num_classes, task_weights=[2.0, 1.8, 1.2, 1.0])
    else:
        raise ValueError(f"Unknown loss type: {loss_type}")

def get_prediction_helper(loss_type):
    if loss_type == "ce":
        return lambda logits: torch.argmax(logits, dim=1)
    elif loss_type in ["corn", "focal_corn"]:
        return corn_label_from_logits
    elif loss_type == "coral":
        return coral_label_from_logits
    else:
        raise ValueError(f"Unknown loss type: {loss_type}")

def get_scheduler(scheduler_type, optimizer, epochs):
    if scheduler_type == "cosine":
        return optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-7)
    elif scheduler_type == "step":
        return optim.lr_scheduler.StepLR(optimizer, step_size=int(epochs*0.33), gamma=0.1)
    elif scheduler_type == "plateau":
        return optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3)
    elif scheduler_type == "paper_step":
        # Manual scheduler
        return None
    else:
        return None

# Instantiate Model
initial_loss = TrainingConfig.loss_stage1 if TrainingConfig.training_pipeline in ["2-stage", "3-stage", "paper_2stage"] else TrainingConfig.loss_standard
model = SEResNeXt50Model(num_classes=5, pretrained=TrainingConfig.pretrained, loss_type=initial_loss)

# Define stage checkpoints paths
stage2_best_path = os.path.join(CHECKPOINT_SAVE_DIR, "stage2_best_model.pth")
stage3_best_path = os.path.join(CHECKPOINT_SAVE_DIR, "best_model.pth")
last_model_path = os.path.join(CHECKPOINT_SAVE_DIR, "last_model.pth")

print(f"SE-ResNeXt-50 Model initialized with {initial_loss.upper()} output configuration!")

## 9. Execute pipeline
Decide which pipeline will be used ( 1,2 or 3 stage)

In [ ]:
best_val_qwk = -1.0
best_val_qwk_stage2 = -1.0
history = []
start_epoch = 0

# Check if we should resume from last checkpoint automatically
resumed = False
stage2_counter = 0
stage3_counter = 0
skip_stage2_flag = False

if getattr(TrainingConfig, "resume_from_last", True) and os.path.exists(last_model_path):
    try:
        print(f"Loading last checkpoint from: {last_model_path} to resume training...")
        try:
            checkpoint = torch.load(last_model_path, map_location=device, weights_only=False)
        except TypeError:
            checkpoint = torch.load(last_model_path, map_location=device)
            
        # Load model weights
        model.load_state_dict(checkpoint['model_state_dict'])
        
        # Load metrics and training history
        history = checkpoint.get('history', [])
        start_epoch = checkpoint.get('epoch', 0)
        best_val_qwk = checkpoint.get('best_val_qwk', -1.0)
        best_val_qwk_stage2 = checkpoint.get('best_val_qwk_stage2', -1.0)
        stage2_counter = checkpoint.get('stage2_counter', 0)
        stage3_counter = checkpoint.get('stage3_counter', 0)
        skip_stage2_flag = checkpoint.get('skip_stage2_flag', False)
        
        print(f"Successfully resumed from Epoch {start_epoch + 1}. Previous best QWK: {best_val_qwk:.4f}")
        resumed = True
    except Exception as e:
        print(f"Could not automatically resume from last checkpoint: {e}. Starting fresh.")

# Execute pipeline based on configuration
pipeline = TrainingConfig.training_pipeline
stage1_epochs = TrainingConfig.stage1_epochs
stage2_epochs = TrainingConfig.stage2_epochs
stage3_epochs = TrainingConfig.stage3_epochs

if pipeline == "paper_2stage":
    total_epochs = stage1_epochs + stage2_epochs
elif pipeline == "3-stage":
    total_epochs = stage1_epochs + stage2_epochs + stage3_epochs
elif pipeline == "2-stage":
    total_epochs = stage1_epochs + stage2_epochs
else:
    total_epochs = TrainingConfig.total_epochs_standard

current_stage = None
optimizer = None
scheduler = None

# Main Loop
for epoch in range(start_epoch, total_epochs):
    # If early stopping triggered transition, skip remaining epochs of Stage 2
    if pipeline == "3-stage" and skip_stage2_flag and epoch < stage1_epochs + stage2_epochs:
        continue

    # Determine stage and configure optimizer dynamically based on epoch index
    if pipeline == "paper_2stage":
        if epoch < stage1_epochs:
            stage_name = "Stage 1"
            loss_type = TrainingConfig.loss_stage1
            if current_stage != "Stage 1":
                print("\n=== CONFIGURING STAGE 1 (Paper Warm-up): Freeze Backbone, Train Heads Only ===")
                model.freeze_backbone()
                # Train only classifier parameters (Adam, LR 1e-3, WD 1e-4)
                head_params = list(model.fc_kl.parameters())
                optimizer = optim.Adam(head_params, lr=TrainingConfig.lr_warmup, weight_decay=TrainingConfig.weight_decay)
                current_stage = "Stage 1"
        else:
            stage_name = "Stage 2"
            loss_type = TrainingConfig.loss_stage2
            if current_stage != "Stage 2":
                print("\n=== CONFIGURING STAGE 2 (Paper Fine-tuning): Unfreeze Backbone, Train All Layers ===")
                model.unfreeze_backbone()
                optimizer = optim.Adam(model.parameters(), lr=TrainingConfig.lr_finetune, weight_decay=TrainingConfig.weight_decay)
                current_stage = "Stage 2"
            
            # Paper custom scheduler: drop learning rate by a factor of 10 at epoch 15 (index 14)
            if epoch == 14:
                for param_group in optimizer.param_groups:
                    param_group['lr'] = param_group['lr'] * 0.1
                print(f"\n--> [PAPER LR SCHEDULER] Dropped learning rate 10x. New LR: {optimizer.param_groups[0]['lr']:.7f}")

    elif pipeline == "3-stage":
        if epoch < stage1_epochs:
            stage_name = "Stage 1"
            loss_type = TrainingConfig.loss_stage1
            if current_stage != "Stage 1":
                print("\n=== CONFIGURING STAGE 1: WARM-UP FC (Backbone Frozen) ===")
                model.freeze_backbone()
                head_params = list(model.fc_kl.parameters())
                optimizer = optim.AdamW(head_params, lr=TrainingConfig.lr_warmup, weight_decay=TrainingConfig.weight_decay)
                current_stage = "Stage 1"
        elif epoch < stage1_epochs + stage2_epochs:
            stage_name = "Stage 2"
            loss_type = TrainingConfig.loss_stage2
            if current_stage != "Stage 2":
                print("\n=== CONFIGURING STAGE 2: COARSE-TUNING (Last Block Unfrozen, Balanced Data) ===")
                model.unfreeze_backbone()
                optimizer = optim.AdamW([
                    {'params': model.backbone.parameters(), 'lr': TrainingConfig.lr_coarse_backbone},
                    {'params': model.fc_kl.parameters(), 'lr': TrainingConfig.lr_coarse_head}
                ], weight_decay=TrainingConfig.weight_decay)
                scheduler = get_scheduler(TrainingConfig.scheduler_stage2, optimizer, stage2_epochs)
                if resumed and checkpoint.get('scheduler_state_dict') is not None:
                    try:
                        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
                    except:
                        pass
                current_stage = "Stage 2"
        else:
            stage_name = "Stage 3"
            loss_type = TrainingConfig.loss_stage3
            if current_stage != "Stage 3":
                print("\n=== CONFIGURING STAGE 3: FINE-TUNING (Load Stage 2 Best, Moderate Sampler, Focal Loss) ===")
                if not resumed or epoch == stage1_epochs + stage2_epochs:
                    if os.path.exists(stage2_best_path):
                        print("Loading best Stage 2 model weights...")
                        checkpoint_s2 = torch.load(stage2_best_path, map_location=device)
                        model.load_state_dict(checkpoint_s2['model_state_dict'])
                model.unfreeze_backbone()
                optimizer = optim.AdamW(model.parameters(), lr=TrainingConfig.lr_finetune, weight_decay=10*TrainingConfig.weight_decay)
                scheduler = get_scheduler(TrainingConfig.scheduler_stage3, optimizer, stage3_epochs)
                if resumed and checkpoint.get('scheduler_state_dict') is not None:
                    try:
                        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
                    except:
                        pass
                current_stage = "Stage 3"
                
    elif pipeline == "2-stage":
        if epoch < stage1_epochs:
            stage_name = "Stage 1"
            loss_type = TrainingConfig.loss_stage1
            if current_stage != "Stage 1":
                print("\n=== CONFIGURING STAGE 1: WARM-UP FC (Backbone Frozen) ===")
                model.freeze_backbone()
                head_params = list(model.fc_kl.parameters())
                optimizer = optim.AdamW(head_params, lr=TrainingConfig.lr_warmup, weight_decay=TrainingConfig.weight_decay)
                current_stage = "Stage 1"
        else:
            stage_name = "Stage 2"
            loss_type = TrainingConfig.loss_stage2
            if current_stage != "Stage 2":
                print("\n=== CONFIGURING STAGE 2: FULL FINE-TUNING (All Layers Unfrozen) ===")
                model.unfreeze_backbone()
                optimizer = optim.AdamW(model.parameters(), lr=TrainingConfig.lr_coarse_head, weight_decay=TrainingConfig.weight_decay)
                scheduler = get_scheduler(TrainingConfig.scheduler_stage2, optimizer, stage2_epochs)
                if resumed and checkpoint.get('scheduler_state_dict') is not None:
                    try:
                        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
                    except:
                        pass
                current_stage = "Stage 2"
                
    else:
        stage_name = "Standard"
        loss_type = TrainingConfig.loss_standard
        if current_stage != "Standard":
            print("\n=== CONFIGURING STANDARD FINE-TUNING ===")
            model.unfreeze_backbone()
            optimizer = optim.Adam(model.parameters(), lr=TrainingConfig.lr_standard, weight_decay=TrainingConfig.weight_decay)
            scheduler = get_scheduler(TrainingConfig.scheduler_standard, optimizer, total_epochs)
            if resumed and checkpoint.get('scheduler_state_dict') is not None:
                try:
                    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
                except:
                    pass
            current_stage = "Standard"
            
            # Manual LR drop at epoch 15 if paper step scheduler is selected for standard
            if TrainingConfig.scheduler_standard == "paper_step" and epoch >= 14:
                for param_group in optimizer.param_groups:
                    param_group['lr'] = TrainingConfig.lr_standard * 0.1

    # Select Dataloader
    active_loader = train_loader
    
    # Fit & Evaluate
    train_loss, train_acc = model.fit(epoch, active_loader, optimizer, loss_type, device)
    val_metrics = model.evaluate(epoch, val_loader, loss_type, device, description="VALIDATE")
    
    val_loss = val_metrics["loss"]
    val_acc = val_metrics["acc"]
    val_report = val_metrics["report"]
    val_qwk = val_metrics["qwk"]
    
    if scheduler is not None:
        if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau):
            scheduler.step(val_loss)
        else:
            scheduler.step()
        current_lr = scheduler.get_last_lr()[0]
    else:
        current_lr = optimizer.param_groups[0]['lr']
        
    print(f"\n--- {stage_name} Epoch {epoch+1}/{total_epochs} ---")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% | Val QWK: {val_qwk:.4f} | LR: {current_lr:.7f}")
    
    history.append({
        "stage": stage_name, "epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc,
        "val_loss": val_loss, "val_acc": val_acc, "qwk": val_qwk, "auc": val_metrics["auc"], "ap": val_metrics["ap"]
    })
    
    # Prepare checkpoint dictionary
    checkpoint_state = {
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
        'best_val_qwk': best_val_qwk,
        'best_val_qwk_stage2': best_val_qwk_stage2,
        'history': history,
        'stage2_counter': stage2_counter,
        'stage3_counter': stage3_counter,
        'skip_stage2_flag': skip_stage2_flag
    }
    
    # Save best Stage 2 model
    if stage_name == "Stage 2":
        if val_qwk > best_val_qwk_stage2:
            best_val_qwk_stage2 = val_qwk
            checkpoint_state['best_val_qwk_stage2'] = best_val_qwk_stage2
            torch.save(checkpoint_state, stage2_best_path)
            print(f"--> Saved best Stage 2 model checkpoint with QWK: {best_val_qwk_stage2:.4f}")
            stage2_counter = 0
        else:
            stage2_counter += 1
            if getattr(TrainingConfig, "use_early_stopping", False) and stage2_counter >= getattr(TrainingConfig, "early_stopping_patience_stage2", 15):
                print(f"Early Stopping Stage 2 triggered. Transitioning to Stage 3!")
                skip_stage2_flag = True
                checkpoint_state['skip_stage2_flag'] = True
                checkpoint_state['epoch'] = stage1_epochs + stage2_epochs
                torch.save(checkpoint_state, last_model_path)
        
    # Save final best model and check early stopping
    if stage_name in ["Stage 2", "Stage 3", "Standard"] and pipeline == "paper_2stage":
        if val_qwk > best_val_qwk:
            best_val_qwk = val_qwk
            checkpoint_state['best_val_qwk'] = best_val_qwk
            torch.save(checkpoint_state, stage3_best_path)
            print(f"--> Saved best final model checkpoint with QWK: {best_val_qwk:.4f}")
            stage3_counter = 0
        else:
            stage3_counter += 1
            if getattr(TrainingConfig, "use_early_stopping", False) and stage3_counter >= getattr(TrainingConfig, "early_stopping_patience_stage3", 15):
                print(f"Early Stopping standard triggered. Training completed!")
                torch.save(checkpoint_state, last_model_path)
                break
        
    # Always save last model checkpoint to support auto-resuming after disconnections
    torch.save(checkpoint_state, last_model_path)
    
    # Disable resumed flag after first running epoch
    resumed = False

# Print final training history log summary table
print("\n" + "="*95)
print("TRAINING HISTORY LOG SUMMARY")
print("="*95)
print(f"{'Stage':<9} | {'Epoch':<5} | {'Train Loss':<10} | {'Train Acc':<9} | {'Val Loss':<8} | {'Val Acc':<7} | {'QWK':<6} | {'ROC AUC':<7} | {'AP':<6}")
print("-"*95)
for h in history:
    print(f"{h['stage']:<9} | {h['epoch']:<5} | {h['train_loss']:<10.4f} | {h['train_acc']:<8.2f}% | {h['val_loss']:<8.4f} | {h['val_acc']:<6.2f}% | {h['qwk']:<6.4f} | {h['auc']:<7.4f} | {h['ap']:<6.4f}")
print("="*95)

## 10. Evaluation on the Test Split
This section loads the independent test split folder from the dataset and evaluates the trained model on it.


In [ ]:
# Load the test dataset split
test_dataset = KaggleKneeOsteoarthritisDataset(
    root=DATASET_ROOT_PATH, split_dir="test", transform=val_loader_transform, exclude_hashes=train_hashes
)
test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Loaded test dataset containing {len(test_dataset)} images.")

# Load the best model weights
if os.path.exists(stage3_best_path):
    print("Loading best model checkpoint for testing...")
    try:
        checkpoint = torch.load(stage3_best_path, map_location=device, weights_only=False)
    except TypeError:
        checkpoint = torch.load(stage3_best_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
else:
    print("Best model checkpoint not found. Testing with current weights.")

# Run evaluation on test dataset
model.to(device)
model.eval()
all_preds, all_labels, all_probas = [], [], []

loss_type = TrainingConfig.loss_stage2 if TrainingConfig.training_pipeline == "paper_2stage" else TrainingConfig.loss_standard
predict_fn = get_prediction_helper(loss_type)

if loss_type == "ce":
    probas_fn = lambda x: F.softmax(x, dim=1)
elif loss_type in ["corn", "focal_corn"]:
    probas_fn = corn_probas
elif loss_type == "coral":
    probas_fn = torch.sigmoid

with torch.no_grad():
    for images, labels in tqdm.tqdm(test_loader, desc="TEST EVALUATION"):
        labels = labels.to(device)
        
        # Support FiveCrop TTA if active
        if len(images.shape) == 5:
            bs, n_crops, c, h, w = images.size()
            flat_images = images.view(-1, c, h, w).to(device)
            outputs = model(flat_images)
            
            probas = probas_fn(outputs)
            probas = probas.view(bs, n_crops, -1).mean(dim=1)
            
            if loss_type == "ce":
                predicted = torch.argmax(probas, dim=1)
            elif loss_type in ["corn", "focal_corn"]:
                predicted = torch.argmax(probas, dim=1)
            elif loss_type == "coral":
                predicted = (probas > 0.5).sum(dim=1)
        else:
            images = images.to(device)
            outputs = model(images)
            probas = probas_fn(outputs)
            predicted = predict_fn(outputs)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probas.extend(probas.cpu().numpy())

# Convert to numpy arrays
y_true = np.array(all_labels)
y_pred = np.array(all_preds)
y_probas = np.array(all_probas)

# Pad probas array to 5 classes if it has 4 (for CORAL/CORN) during OVR calculations
if y_probas.shape[1] == 4:
    if loss_type in ["corn", "focal_corn"]:
        pass
    elif loss_type == "coral":
        temp = np.zeros((y_probas.shape[0], 5))
        temp[:, 0] = 1.0 - y_probas[:, 0]
        for idx in range(1, 4):
            temp[:, idx] = y_probas[:, idx - 1] - y_probas[:, idx]
        temp[:, 4] = y_probas[:, 3]
        y_probas = np.clip(temp, 0.0, 1.0)

## 11. Calculate metric
Calculate basic metrics (precision, recall, f1, ..) + 95% CI + UAC + AP


In [ ]:
# 1. Compute basic metrics
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, average_precision_score, confusion_matrix
from torchmetrics.classification import CohenKappa
import torch

test_acc = accuracy_score(y_true, y_pred)
kappa_metric = CohenKappa(task="multiclass", num_classes=5, weights="quadratic")
test_qwk = kappa_metric(torch.tensor(y_pred), torch.tensor(y_true)).item()
test_auc = roc_auc_score(y_true, y_probas, multi_class='ovr', average='macro')
y_one_hot = np.eye(5)[y_true]
test_ap = average_precision_score(y_one_hot, y_probas, average='macro')

# 2. Compute 95% Confidence Intervals using Bootstrapping
print("\nComputing 95% Confidence Intervals via bootstrapping (200 iterations)...")
boot_acc, boot_qwk, boot_auc, boot_ap = [], [], [], []
rng = np.random.default_rng(42)
for _ in range(200):
    indices = rng.choice(len(y_true), size=len(y_true), replace=True)
    if len(np.unique(y_true[indices])) < 5:
        continue
    y_true_b = y_true[indices]
    y_pred_b = y_pred[indices]
    y_probas_b = y_probas[indices]
    
    boot_acc.append(accuracy_score(y_true_b, y_pred_b))
    boot_qwk.append(kappa_metric(torch.tensor(y_pred_b), torch.tensor(y_true_b)).item())
    try:
        boot_auc.append(roc_auc_score(y_true_b, y_probas_b, multi_class='ovr', average='macro'))
    except:
        pass
    try:
        y_one_hot_b = np.eye(5)[y_true_b]
        boot_ap.append(average_precision_score(y_one_hot_b, y_probas_b, average='macro'))
    except:
        pass

def get_ci(data):
    sorted_data = np.sort(data)
    low = sorted_data[int(0.025 * len(sorted_data))]
    high = sorted_data[int(0.975 * len(sorted_data))]
    return low, high

acc_ci = get_ci(boot_acc)
qwk_ci = get_ci(boot_qwk)
auc_ci = get_ci(boot_auc)
ap_ci = get_ci(boot_ap)

print("\n" + "="*50)
print("=== FINAL TEST METRICS WITH 95% CONFIDENCE INTERVALS ===")
print("="*50)
print(f"Accuracy: {test_acc:.4f} (95% CI: {acc_ci[0]:.4f} - {acc_ci[1]:.4f})")
print(f"QWK Score: {test_qwk:.4f} (95% CI: {qwk_ci[0]:.4f} - {qwk_ci[1]:.4f})")
print(f"ROC AUC: {test_auc:.4f} (95% CI: {auc_ci[0]:.4f} - {auc_ci[1]:.4f})")
print(f"Average Precision (AP): {test_ap:.4f} (95% CI: {ap_ci[0]:.4f} - {ap_ci[1]:.4f})")
print("="*50)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=[str(i) for i in range(5)], zero_division=0))

## 12. Draw diagram
Draw confusion matrix, ROC, precision-recall curve

In [ ]:
import seaborn as sns

# 1. Plot Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=[str(i) for i in range(5)], 
            yticklabels=[str(i) for i in range(5)])
plt.title('Confusion Matrix (SE-ResNeXt-50 Paper Method)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_SAVE_DIR, 'confusion_matrix.png'))
plt.show()

# 2. Plot ROC Curves (One-vs-Rest)
from sklearn.metrics import roc_curve, auc
plt.figure(figsize=(8, 6))
for i in range(5):
    fpr, tpr, _ = roc_curve((y_true == i).astype(int), y_probas[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'Class {i} (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) - OVR')
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_SAVE_DIR, 'roc_curve.png'))
plt.show()

# 3. Plot Precision-Recall Curves (One-vs-Rest)
from sklearn.metrics import precision_recall_curve
plt.figure(figsize=(8, 6))
for i in range(5):
    precision, recall, _ = precision_recall_curve((y_true == i).astype(int), y_probas[:, i])
    ap = average_precision_score((y_true == i).astype(int), y_probas[:, i])
    plt.plot(recall, precision, label=f'Class {i} (AP = {ap:.4f})')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve - OVR')
plt.legend(loc="lower left")
plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_SAVE_DIR, 'precision_recall_curve.png'))
plt.show()